In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
import random
from sklearn.preprocessing import StandardScaler
import umap
import hdbscan

In [ ]:
def parse_embedding(x):
    if isinstance(x, str):
        return np.fromstring(x.strip("[]"), sep=' ', dtype=np.float32)
    return x

In [ ]:
emb_sum    = {}
emb_sq_sum = {}
tweet_count = defaultdict(int)
sum_length  = defaultdict(float)
unique_words = defaultdict(set)
sum_hashtags = defaultdict(int)
sum_mentions = defaultdict(int)

chunksize = 50000

for chunk in pd.read_csv(
    "/content/drive/MyDrive/Mathematics_and_AI/results.csv",
    encoding="latin1",
    chunksize=chunksize
):

    chunk["embedding"] = chunk["embedding"].apply(parse_embedding)
    chunk = chunk.dropna(subset=["embedding"])

    for user, emb, text in zip(chunk["user_name"], chunk["embedding"], chunk["cleanText"]):

        txt = str(text)

        # --- Embedding mean & variance ---
        if user not in emb_sum:
            emb_sum[user]    = np.zeros_like(emb)
            emb_sq_sum[user] = np.zeros_like(emb)

        emb_sum[user]    += emb
        emb_sq_sum[user] += emb * emb

        tweet_count[user] += 1

        # --- Text stats ---
        sum_length[user] += len(txt)

        # unique words (store only a small set)
        for w in txt.split():
            if len(unique_words[user]) < 500:  # limit to save RAM
                unique_words[user].add(w)

        sum_hashtags[user] += txt.count("#")
        sum_mentions[user] += txt.count("@")

In [ ]:
# --- 50 % der User auswählen ---
all_users = list(tweet_count.keys())
random.seed(42)
usernames = random.sample(all_users, k=len(all_users) // 2)

feature_vectors = []
embed_mean_list = []
embed_var_list  = []

for user in usernames:
    n = tweet_count[user]

    # Embedding mean
    mean_vec = emb_sum[user] / n

    # Embedding variance
    var_vec = (emb_sq_sum[user] / n) - (mean_vec * mean_vec)

    embed_mean_list.append(mean_vec)
    embed_var_list.append(var_vec)

    # additional numeric features
    f_count      = n
    f_avg_length = sum_length[user] / n
    f_unique     = len(unique_words[user])
    f_hashtags   = sum_hashtags[user] / n
    f_mentions   = sum_mentions[user] / n

    feature_vectors.append([f_count, f_avg_length, f_unique, f_hashtags, f_mentions])

# Convert to matrices
X_embed_mean = np.vstack(embed_mean_list)
X_embed_var  = np.vstack(embed_var_list)
X_num        = np.array(feature_vectors)

In [ ]:
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)

X = np.hstack([X_embed_mean, X_embed_var, X_num_scaled])

In [ ]:
umap_model = umap.UMAP(
    n_neighbors=20,
    min_dist=0.1,
    n_components=2,
    metric="cosine",
    low_memory=True,
    densmap=False
)
X_umap = umap_model.fit_transform(X)

In [ ]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=1500,
    min_samples=300,
    metric='euclidean',
    cluster_selection_method='leaf'
)

labels = clusterer.fit_predict(X_umap)

In [ ]:
df_clusters = pd.DataFrame({
    "user_name": usernames,
    "cluster": labels
})
df_clusters.to_csv("user_clusters.csv", index=False, encoding="latin1")